# Kiểm thử tải trên Kaggle CPU

Notebook này thực hiện kế hoạch kiểm thử tải trong [`docs/load-testing.md`](https://github.com/24020196/BTL-KTPM/blob/main/docs/load-testing.md) và sinh báo cáo hoàn thành kiểm thử `REPORT.md` cho từng phiên bản.
Mã nguồn bộ đo nằm trong thư mục `loadtest/`; notebook chỉ điều phối và hiển thị kết quả.

Các bước:

1. Đặt tham số.
2. Cài đặt môi trường.
3. Khởi động MySQL và ghi lại cấu hình máy.
4. Xem kế hoạch kiểm thử sẽ chạy.
5. Chạy kế hoạch cho từng phiên bản.
6. Xem báo cáo của từng phiên bản.
7. So sánh hai phiên bản, nếu có.
8. Đóng gói kết quả.

## 1. Tham số

| Tham số | Ý nghĩa |
| :--- | :--- |
| `REPO_URL` | Repo chứa mã nguồn; nhóm làm Phase 2 đổi thành repo của nhóm |
| `HARNESS_REF` | Nhánh hoặc mã commit chứa phiên bản bộ đo; đo đường cơ sở Phase 1 dùng `"main"`, nhóm làm Phase 2 dùng mã commit Phase 1 |
| `REFS` | Các phiên bản ứng dụng cần đo, theo thứ tự, dạng `{"nhãn": "nhánh, tag hoặc commit"}` |
| `QUICK` | `True` để chạy thử khoảng 5 phút mỗi phiên bản; `False` cho lần đo chính thức, khoảng 30 phút mỗi phiên bản |
| `OUT` | Thư mục chứa kết quả |

Ví dụ khi đối chiếu Phase 2 với đường cơ sở Phase 1: `REFS = {"phase1": "<commit Phase 1>", "phase2": "main"}`, với `<commit Phase 1>` lấy trong `docs/benchmark/phase1/REPORT.md`. Quy định về phần được thay đổi trong bộ đo nằm ở mục 8 của kế hoạch kiểm thử.

In [ ]:
REPO_URL = "https://github.com/24020196/BTL-KTPM.git"   # nhóm làm Phase 2: đổi thành repo của nhóm
HARNESS_REF = "main"            # nhóm làm Phase 2: mã commit Phase 1
REFS = {"phase1": "HEAD"}       # nhóm làm Phase 2: {"phase1": "<commit Phase 1>", "phase2": "<nhánh>"}
QUICK = False
OUT = "/kaggle/working/results"

## 2. Cài đặt môi trường

In [ ]:
!apt-get update -qq > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-21-jdk-headless maven mysql-server psmisc > /dev/null 2>&1
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java > /dev/null 2>&1
!pip install -q locust psutil pymysql requests pandas matplotlib
!java -version 2>&1 | head -1; mysqld --version; locust --version

In [ ]:
import os, sys, subprocess
REPO_DIR = "/kaggle/working/BTL-KTPM"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags", "-q"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", HARNESS_REF], check=True)
sys.path.insert(0, f"{REPO_DIR}/loadtest")
os.makedirs(OUT, exist_ok=True)
import bench, analyze, run_plan
print("harness commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout)

## 3. Khởi động MySQL và ghi lại cấu hình máy

In [ ]:
import json, pandas as pd
mysql_info = bench.start_mysql()
env = None
for label, ref in REFS.items():
    env = bench.capture_env(f"{OUT}/{label}", REPO_DIR, {label: ref})
env["mysql_settings"] = mysql_info
display(pd.DataFrame([(k, json.dumps(v) if isinstance(v, (dict, list)) else v) for k, v in env.items()], columns=["item", "value"]))

## 4. Kế hoạch kiểm thử

Bảng dưới đây liệt kê các bài sẽ chạy, theo đúng thứ tự. Lý do chọn từng tham số và tiêu chí đạt được mô tả ở mục 4 và mục 5 của kế hoạch kiểm thử.

In [ ]:
plan = run_plan.build_plan(quick=QUICK)
bench.write_json(f"{OUT}/plan.json", plan)
pd.DataFrame([
    {
        "Bài": name,
        "Người dùng ảo": cfg.get("users"),
        "Thời lượng (s)": cfg.get("duration"),
        "Các bậc [giây kết thúc, người dùng ảo, tốc độ tăng]": cfg.get("shape"),
        "Thời gian nghỉ (s)": cfg["wait"],
        "Số lần lặp": cfg.get("repetitions", 1),
    }
    for name, cfg in plan.items()
])

## 5. Chạy kế hoạch cho từng phiên bản
Với mỗi phiên bản, bộ đo build ứng dụng từ commit tương ứng, tạo CSDL mới, nạp dữ liệu, chạy lần lượt các bài kiểm thử rồi dừng ứng dụng. Nếu bài smoke không đạt, bộ đo dừng và in lý do.

In [ ]:
summaries = {}
for label, ref in REFS.items():
    summaries[label] = run_plan.run_label(label, ref, REPO_DIR, OUT, plan)

## 6. Báo cáo từng phiên bản
Báo cáo của mỗi phiên bản nằm ở `results/<nhãn>/REPORT.md`, kèm các biểu đồ PNG.

In [ ]:
from IPython.display import Markdown, Image
for label in REFS:
    report = open(f"{OUT}/{label}/REPORT.md", encoding="utf-8").read()
    display(Markdown(chr(10).join(l for l in report.splitlines() if not l.startswith("!["))))
    for img in ("breakpoint.png", "spike.png", "endpoints_average_load.png"):
        p = f"{OUT}/{label}/{img}"
        if os.path.exists(p):
            display(Image(p))

## 7. So sánh hai phiên bản
Bước này chỉ chạy khi `REFS` có từ hai phiên bản trở lên. Kết quả nằm ở `results/comparison/COMPARISON.md`.

In [ ]:
labels = list(REFS)
if len(labels) >= 2:
    cmp = analyze.compare(f"{OUT}/{labels[0]}", f"{OUT}/{labels[-1]}", f"{OUT}/comparison")
    display(cmp)
    display(Image(f"{OUT}/comparison/comparison_breakpoint.png"))
else:
    print("Chỉ có 1 phiên bản, bỏ qua so sánh.")

## 8. Đóng gói kết quả
Tải `results.zip` ở tab Output của Kaggle. Chỉ đưa `REPORT.md` và các biểu đồ PNG vào repo (`docs/benchmark/<nhãn>/`); dữ liệu thô giữ ở output của notebook.

In [ ]:
import shutil
for d in os.listdir(OUT):
    if d.startswith("_src_"):
        subprocess.run(["git", "-C", REPO_DIR, "worktree", "remove", "--force", f"{OUT}/{d}"], check=False)
        shutil.rmtree(f"{OUT}/{d}", ignore_errors=True)
shutil.make_archive("/kaggle/working/results", "zip", OUT)
print("saved /kaggle/working/results.zip")